# LightShap — run all

User-facing orchestrator. Scientific logic lives in `src/lightshap/`, not here. Re-running this notebook is safe: completed stages are skipped when their artifacts still validate.

**Estimand.** Inference-time extra-hop inclusion on one frozen uniform `K=3`, `d=64` LightGCN. Skip patterns are cache scoring rules. The flag is not a prune claim. Beauty is **2014** 5-core. `I_12 = 1/2`, never `1`.

## 1. Environment

In [ ]:
from pathlib import Path
import json, sys, platform
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists() and (ROOT.parent / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from lightshap import __version__
from lightshap.constants import (
    FIXTURE_PHI, FIXTURE_I, LIGHTGCN_FROZEN, STABILITY_FLOOR,
    N_VAL_ALPHA, BEAUTY_YEAR, FORBIDDEN_I12,
)
from lightshap.config import load_experiment_config
from lightshap.pipeline.stages import STAGE_ORDER
from lightshap.pipeline.runner import PipelineRunner
from lightshap.pipeline.state import load_state
print('lightshap', __version__)
print('python', sys.version.split()[0], platform.platform())
print('repo', ROOT)
print('stages', len(STAGE_ORDER))
from lightshap.utils.device import resolve_device
print('device', resolve_device('auto'))
try:
    import torch
    mps = bool(getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available())
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'mps', mps)
except Exception as exc:
    print('torch missing:', exc)


## 2. Frozen invariants (must not be edited for convenience)

In [ ]:
print('LightGCN', LIGHTGCN_FROZEN)
print('stability floor', STABILITY_FLOOR)
print('val-alpha candidates', N_VAL_ALPHA)
print('fixture phi', FIXTURE_PHI, 'I', FIXTURE_I, 'forbidden I12', FORBIDDEN_I12)
print('Beauty year', BEAUTY_YEAR)
assert abs(FIXTURE_I - 0.5) < 1e-15
assert abs(FIXTURE_I - 1.0) > 1e-9


## 3. Configuration

On a Mac M4 Pro start with `synthetic`, then set `PROFILE = 'experiment'` for ML-1M + Beauty 2014. `device: auto` selects MPS on Apple Silicon. Paste SHA256s from `python scripts/fetch_data.py` before a scientific run.

In [ ]:
# synthetic_fast | synthetic | experiment
PROFILE = 'synthetic'
CONFIG = ROOT / 'configs' / f'{PROFILE}.yaml'
cfg = load_experiment_config(CONFIG, project_root=ROOT)
print('profile', cfg.profile)
print('config', CONFIG)
print('datasets', cfg.datasets)
print('seeds', cfg.seeds)
print('neg_pool', cfg.neg_pool)
print('device requested', cfg.device, '->', resolve_device(cfg.device))
print('config_hash', cfg.config_hash())
print('LightGCN', cfg.lightgcn)


## 4. Dataset / run status

In [ ]:
runs_root = ROOT / 'results' / 'runs'
existing = sorted(p.name for p in runs_root.glob('*') if p.is_dir()) if runs_root.exists() else []
print('existing runs:', existing[-5:])
processed = ROOT / 'data' / 'processed'
print('processed datasets:', [p.name for p in processed.glob('*')] if processed.exists() else [])


## 5. Run the pipeline

The runner refuses to execute a stage whose dependencies are missing. Resume semantics: completed+valid stages are not recomputed.

In [ ]:
RESUME = True  # safe re-run
runner = PipelineRunner(cfg)
print('run_id', runner.run_id)
print('run_dir', runner.run_dir)
status = runner.run(resume=RESUME)
print('finished', status['run_id'])


## 6. Checkpoint / stage table

In [ ]:
st = load_state(runner.run_dir)
for name, rec in st.stages.items():
    err = f'  ({rec.error})' if rec.error else ''
    print(f'{name:24} {rec.state}{err}')


## 7. Results: RQ1–RQ4

In [ ]:
def loadj(rel):
    p = runner.run_dir / rel
    return json.loads(p.read_text()) if p.is_file() else None

rq1 = loadj('rq1/rq1.json')
rq2 = loadj('rq2/rq2.json')
rq3 = loadj('rq3/rq3.json')
rq4 = loadj('rq4/rq4.json')
shap = loadj('shapley/summary.json')

if shap:
    for ds, rows in shap.items():
        print('==', ds, 'Game A ==')
        for row in rows:
            print(' seed', row['seed'], 'phi', row['phi'], 'v(L)', row['v'].get('(1, 2, 3)'))
            print(' flag any', row['flag']['any_pair_flagged'], 'undefined', row['flag']['undefined'])
            print(' Game B trigger', row['game_b_trigger']['triggered'])

if rq1:
    for ds, rec in rq1.items():
        print('RQ1', ds, 'v_L_mean', rec.get('v_L_mean'), 'spearman', rec.get('v_C_spearman_mean_pairwise'))
        print('  modal order', rec.get('phi_modal_order'), 'cosine', rec.get('phi_mean_pairwise_cosine'))
        print('  note', rec.get('note'))

if rq3:
    for ds, rec in rq3.items():
        print('RQ3', ds, 'mean_delta', rec.get('mean_delta_if_all_eligible'), rec.get('result_state'))

if rq4:
    for ds, rec in rq4.items():
        agg = rec['aggregate']
        print('RQ4', ds, agg.get('mean_test_ndcg'), 'holm', agg.get('holm'))


## 8. Figures

In [ ]:
from IPython.display import Image, display, Markdown
figdir = runner.run_dir / 'figures'
figs = sorted(figdir.glob('*.png')) if figdir.exists() else []
print(len(figs), 'figures')
for p in figs:
    display(Markdown(f'**{p.name}**'))
    display(Image(filename=str(p)))


## 9. Warnings and compliance (honest — no fabricated PASS)

In [ ]:
from collections import Counter
summary = loadj('summary.json')
print('run status', summary.get('status') if summary else None)
print('warnings:')
for w in (summary or {}).get('warnings', []) or ['_none_']:
    print(' -', w)
comp = loadj('spec_compliance.json')
if comp:
    counts = Counter(v.get('status') for v in comp.values() if isinstance(v, dict) and 'status' in v)
    print('compliance counts', dict(counts))
    print()
    for k, v in sorted(comp.items()):
        if not isinstance(v, dict) or 'status' not in v:
            continue
        print(f"{v['status']:16} {k:22} {v.get('name', '')}")


## 10. Resume after interruption

Re-run the next cell to continue the same `run_id`. Completed stages stay completed.

In [ ]:
status2 = runner.run(resume=True)
print('resume status ok, run_id', status2['run_id'])
